In [1]:
import pandas as pd

# NOTE: merchant_fraud_rate, channel_risk_score, persona_fraud_risk,
# location_fraud_risk are deliberately excluded here - these are pre-computed
# "someone else's risk score" columns, which is target leakage per the PRD's
# explicit rule. See docs/updates for the full reasoning.
usecols = [
    'is_fraud', 'fraud_type', 'source',
    'spending_deviation_score', 'velocity_score', 'geo_anomaly_score', 'amount_ngn',
    'geospatial_velocity_anomaly', 'txn_hour', 'is_weekend', 'is_salary_week', 'is_night_txn',
    'device_seen_count', 'is_device_shared', 'ip_seen_count', 'is_ip_shared',
    'user_txn_count_total', 'user_avg_txn_amt', 'user_std_txn_amt', 'user_txn_frequency_24h',
    'txn_count_last_1h', 'txn_count_last_24h', 'total_amount_last_1h',
    'time_since_last', 'avg_gap_between_txns',
    'is_high_risk_state', 'is_high_risk_lga', 'is_ato_risk',
]
dtype_map = {
    'is_fraud': 'bool', 'fraud_type': 'category', 'source': 'category',
    'amount_ngn': 'float32', 'spending_deviation_score': 'float32',
    'velocity_score': 'int32', 'geo_anomaly_score': 'float32',
    'txn_hour': 'int8', 'is_weekend': 'int8', 'is_salary_week': 'int8', 'is_night_txn': 'int8',
    'device_seen_count': 'int32', 'is_device_shared': 'int8', 'ip_seen_count': 'int32', 'is_ip_shared': 'int8',
    'user_txn_count_total': 'int32', 'user_avg_txn_amt': 'float32', 'user_std_txn_amt': 'float32',
    'user_txn_frequency_24h': 'int32', 'txn_count_last_1h': 'int32', 'txn_count_last_24h': 'int32',
    'total_amount_last_1h': 'float32', 'time_since_last': 'float32', 'avg_gap_between_txns': 'float32',
    'is_high_risk_state': 'int8', 'is_high_risk_lga': 'int8', 'is_ato_risk': 'int8',
    'geospatial_velocity_anomaly': 'bool',
}
df = pd.read_csv('../data/processed/nigerian_transactions_clean.csv', usecols=usecols, dtype=dtype_map)
print(df.shape)
print(df['fraud_type'].value_counts(dropna=False))

(5086000, 28)
fraud_type
NaN                        4902946
Account Takeover            173730
Identity Fraud                9312
Impossible Travel Fraud         12
Name: count, dtype: int64


## Fraud pattern mapping vs. PRD

The PRD names three MVP fraud patterns: account compromise, money-mule fan-out,
and card testing. Our dataset's real `fraud_type` labels are different:

| Dataset label | PRD pattern match | Real row count |
|---|---|---|
| Account Takeover | Account compromise (close match) | ~173,730 |
| Identity Fraud | No PRD match | ~9,312 |
| Impossible Travel Fraud | No PRD match | 12 |

Money-mule fan-out and card testing do not exist in this dataset. This is a
disclosed limitation of using a pre-existing public dataset rather than a
custom-built simulator, and is documented here rather than glossed over.

In [2]:
from sklearn.model_selection import train_test_split
import gc

df_sample, _ = train_test_split(df, train_size=1_000_000, random_state=42, stratify=df['is_fraud'])
del df
gc.collect()

FEATURES = [
    'spending_deviation_score', 'velocity_score', 'geo_anomaly_score', 'amount_ngn',
    'geospatial_velocity_anomaly', 'txn_hour', 'is_weekend', 'is_salary_week', 'is_night_txn',
    'device_seen_count', 'is_device_shared', 'ip_seen_count', 'is_ip_shared',
    'user_txn_count_total', 'user_avg_txn_amt', 'user_std_txn_amt', 'user_txn_frequency_24h',
    'txn_count_last_1h', 'txn_count_last_24h', 'total_amount_last_1h',
    'time_since_last', 'avg_gap_between_txns',
    'is_high_risk_state', 'is_high_risk_lga', 'is_ato_risk',
]

X = df_sample[FEATURES]
y = df_sample['is_fraud']
fraud_type = df_sample['fraud_type']

X_train, X_test, y_train, y_test, ft_train, ft_test = train_test_split(
    X, y, fraud_type, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, " Fraud rate:", y_train.mean())
print("Test:", X_test.shape, " Fraud rate:", y_test.mean())

Train: (800000, 25)  Fraud rate: 0.0359925
Test: (200000, 25)  Fraud rate: 0.03599


In [3]:
from sklearn.ensemble import HistGradientBoostingClassifier
import time

model = HistGradientBoostingClassifier(
    max_iter=200,
    max_depth=8,
    learning_rate=0.1,
    class_weight='balanced',
    random_state=42,
    verbose=1
)

start = time.time()
model.fit(X_train, y_train)
print(f"\nTraining took {time.time() - start:.1f} seconds")

Binning 0.144 GB of training data: 

0.515 s
Binning 0.016 GB of validation data: 

0.019 s
Fitting gradient boosted rounds:


Fit 54 trees in 4.327 s, (1674 total leaves)
Time spent computing histograms: 1.209s
Time spent finding best splits:  0.044s
Time spent applying splits:      0.270s
Time spent predicting:           0.064s

Training took 4.4 seconds


## NOT USED GOING FORWARD

The cell below evaluates the model at the default 0.5 classification threshold.
Result: recall 0.95 but precision only 0.04, with a false-positive burden of
771 per 1,000 legitimate transactions - unusable for a real analyst queue.
This is the same `class_weight='balanced'` + default-threshold overtriggering
problem we diagnosed with the earlier Random Forest attempt.

Kept here for the record, but **superseded by the threshold-sweep cell below**,
which finds a usable operating point instead of relying on the default 0.5 cut.

In [4]:
# SUPERSEDED - see markdown note above. Kept for the record only.
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("=== Held-out test set performance at default threshold 0.5 (NOT USED) ===")
print(classification_report(y_test, y_pred, target_names=['Not Fraud', 'Fraud']))

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)

fp = cm[0][1]
total_legit = cm[0].sum()
print(f"\nFalse positives: {fp} out of {total_legit} legitimate transactions")
print(f"False-positive burden: {fp/total_legit*1000:.2f} false alerts per 1,000 legitimate transactions")

=== Held-out test set performance at default threshold 0.5 (NOT USED) ===
              precision    recall  f1-score   support

   Not Fraud       0.99      0.23      0.37    192802
       Fraud       0.04      0.95      0.08      7198

    accuracy                           0.25    200000
   macro avg       0.52      0.59      0.23    200000
weighted avg       0.96      0.25      0.36    200000



Confusion matrix:
 [[ 44131 148671]
 [   365   6833]]

False positives: 148671 out of 192802 legitimate transactions
False-positive burden: 771.11 false alerts per 1,000 legitimate transactions


## Final evaluation - threshold sweep (this is the one we use)

Instead of the default 0.5 cutoff, we sweep thresholds to find the trade-off
between recall, precision, and false-positive burden - matching the PRD's
"alert budget" concept (section 8.9).

In [5]:
from sklearn.metrics import precision_recall_curve, precision_score, recall_score
import numpy as np

y_proba = model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores)
print(f"Best F1 threshold: {thresholds[best_idx]:.4f}")
print(f"At that threshold - Precision: {precisions[best_idx]:.4f}, Recall: {recalls[best_idx]:.4f}")

print("\nThreshold sweep - precision, recall, and false-positive burden per 1,000:")
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    preds_at_t = (y_proba >= t).astype(int)
    p = precision_score(y_test, preds_at_t, zero_division=0)
    r = recall_score(y_test, preds_at_t, zero_division=0)
    cm_t = confusion_matrix(y_test, preds_at_t)
    fp_t = cm_t[0][1]
    total_legit_t = cm_t[0].sum()
    burden = fp_t / total_legit_t * 1000
    n_alerts = preds_at_t.sum()
    print(f"Threshold {t}: precision={p:.3f}, recall={r:.3f}, FP burden={burden:.1f}/1000, total alerts={n_alerts}")

Best F1 threshold: 0.5081
At that threshold - Precision: 0.0440, Recall: 0.9378

Threshold sweep - precision, recall, and false-positive burden per 1,000:
Threshold 0.3: precision=0.044, recall=0.999, FP burden=813.1/1000, total alerts=163956
Threshold 0.4: precision=0.044, recall=0.993, FP burden=809.2/1000, total alerts=163174
Threshold 0.5: precision=0.044, recall=0.949, FP burden=771.1/1000, total alerts=155504


Threshold 0.6: precision=0.052, recall=0.007, FP burden=4.5/1000, total alerts=909
Threshold 0.7: precision=0.086, recall=0.002, FP burden=0.7/1000, total alerts=151
Threshold 0.8: precision=0.000, recall=0.000, FP burden=0.0/1000, total alerts=0
Threshold 0.9: precision=0.000, recall=0.000, FP burden=0.0/1000, total alerts=0


## Unseen-typology test

Per the PRD (section 9.5), train the model with one fraud pattern held out
entirely, then test whether it can still catch that pattern using only what
it learned from the other fraud types plus normal behaviour. This tells us
whether the model generalises to a "new" kind of fraud it never saw an
example of, rather than just memorising each labelled type.

Run only for **Account Takeover** (173,730 real examples - enough to hold
out and still train meaningfully) and **Identity Fraud** (9,312 examples -
smaller, but still statistically usable). **Impossible Travel Fraud is
skipped** - only 12 examples exist in the entire dataset, too few to hold
out and report an honest recall figure.

In [6]:
def unseen_typology_test(df_sample, features, holdout_type, random_state=42):
    """
    Train with ALL rows of `holdout_type` fraud removed from training data,
    then measure recall on those held-out rows specifically.
    """
    is_holdout_fraud = df_sample['fraud_type'] == holdout_type

    # Training pool: everything except the held-out fraud type's rows
    train_pool = df_sample[~is_holdout_fraud]
    # Test pool: a fresh sample of normal traffic + ALL of the held-out fraud type
    holdout_fraud_rows = df_sample[is_holdout_fraud]

    X_train_ut = train_pool[features]
    y_train_ut = train_pool['is_fraud']

    model_ut = HistGradientBoostingClassifier(
        max_iter=200, max_depth=8, learning_rate=0.1,
        class_weight='balanced', random_state=random_state
    )
    model_ut.fit(X_train_ut, y_train_ut)

    X_holdout = holdout_fraud_rows[features]
    y_holdout_pred = model_ut.predict(X_holdout)

    recall = y_holdout_pred.mean()  # since every row here IS fraud, this equals recall
    n = len(holdout_fraud_rows)
    n_caught = y_holdout_pred.sum()

    print(f"=== Unseen-typology test: {holdout_type} ===")
    print(f"Held-out fraud examples (never in training): {n}")
    print(f"Caught by model trained WITHOUT this pattern: {n_caught} ({recall:.4f} recall)")
    print()
    return recall

recall_at = unseen_typology_test(df_sample, FEATURES, 'Account Takeover')
recall_if = unseen_typology_test(df_sample, FEATURES, 'Identity Fraud')

print("Summary:")
print(f"Account Takeover (unseen): {recall_at:.4f}")
print(f"Identity Fraud (unseen):   {recall_if:.4f}")
print("Impossible Travel Fraud: SKIPPED - only 12 real examples, too few to test honestly")

=== Unseen-typology test: Account Takeover ===
Held-out fraud examples (never in training): 34179
Caught by model trained WITHOUT this pattern: 20990 (0.6141 recall)



=== Unseen-typology test: Identity Fraud ===
Held-out fraud examples (never in training): 1812
Caught by model trained WITHOUT this pattern: 1735 (0.9575 recall)

Summary:
Account Takeover (unseen): 0.6141
Identity Fraud (unseen):   0.9575
Impossible Travel Fraud: SKIPPED - only 12 real examples, too few to test honestly
